# Step 7 – Proliferation module and figure data (Sections 6.1–6.6, Table 14)

Proliferation-module enrichment of the selected genes and the data behind the SHAP and permutation figures.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import glob, pickle, numpy as np, pandas as pd
from scipy.stats import spearmanr, fisher_exact
%run pipeline.ipynb
S = pickle.load(open("summary.pkl","rb")); A = S["A"]
CG = list(pd.read_csv("common_genes.csv").iloc[:,0])
C4 = ["GSE7390","UPP","MAINZ","UNT"]
out = {}

### Proliferation module: genes with Spearman rho >= 0.5 with AURKA in each cohort

In [ ]:
# ---- proliferation module: genes with Spearman rho >= 0.5 with AURKA in each cohort
prolif = {}
for c in C4:
    X, y, P = load(c, genes=CG)
    r = X.rank().corrwith(X["AURKA"].rank())
    mod = set(r[r >= 0.5].index); prolif[c] = mod
    fr = A[c]["freq"]; sel = set(fr.index)
    top = set(fr.head(20).index)
    bg = len(mod) / len(CG)
    a = len(sel & mod); b = len(sel) - a
    odds, p = fisher_exact([[a, b], [len(mod) - a, len(CG) - len(mod) - b]], alternative="greater")
    wfrac = fr[[g for g in fr.index if g in mod]].sum() / fr.sum()   # selection-frequency weighted share
    out[c] = dict(module=len(mod), bg=bg, sel=len(sel), sel_in=a, frac=a/len(sel), wfrac=wfrac, odds=odds, p=p,
                  top20_in=len(top & mod))
    print(c, out[c])
# RRA top 20 and LODO genes
R = S["RRA"]; allmod = set.union(*prolif.values()); anymod = lambda g: sum(g in prolif[c] for c in C4) >= 2
print("RRA top20 in module (>=2 cohorts):", sum(anymod(g) for g in R.index[:20]), list(R.index[:20]))
lodo_g = [g for c in C4 for g in S["LODO"][c]["genes"]]
print("LODO genes in module (>=2 cohorts):", sum(anymod(g) for g in lodo_g), len(lodo_g))
print("pub genes in module:", {g: [g in prolif[c] for c in C4] for g in ["E2F3","PSMC3IP","GINS1","PLAGL2"]})
print("pub genes freq:", {g: [round(A[c]['freq'].get(g,0),2) for c in C4] for g in ["E2F3","PSMC3IP","GINS1","PLAGL2"]})
out["rra_top20_mod"] = sum(anymod(g) for g in R.index[:20]); out["lodo_mod"] = (sum(anymod(g) for g in lodo_g), len(lodo_g))

### SHAP summaries for GSE7390

In [ ]:
# ---- SHAP summaries for GSE7390
F = [pickle.load(open(f,"rb")) for f in sorted(glob.glob("folds_GSE7390/r0_k*.pkl"))]
sm = A["GSE7390"]["shap_mean"]; tot = sm.sum(1).sort_values(ascending=False)
top = list(tot.index[:8])
g2share = (sm["G2"] / sm.sum(1))
print("SHAP top8 mean|phi| x100:\n", (100*sm.loc[top]).round(2), "\nG2 share", g2share.loc[top].round(2).to_dict())
# direction: corr between bin and phi_G3 / phi_G1 among tumours where gene in model
dirn = {}; bees = {}
for g in top:
    b, p3, p1 = [], [], []
    for f in F:
        if g in list(f["genes"]):
            j = list(f["genes"]).index(g); nb = f["Dte"][:, j]
            b += list(nb); p3 += list(f["shap"][:, j, 2]); p1 += list(f["shap"][:, j, 0])
    b = np.array(b); 
    dirn[g] = (len(b), spearmanr(b, p3).statistic, spearmanr(b, p1).statistic, int(b.max()) + 1)
    bees[g] = list(zip(b.tolist(), np.round(p3, 4).tolist()))
print("direction (n, rho bin~phiG3, rho bin~phiG1, nbins):", {g: (v[0], round(v[1],2), round(v[2],2), v[3]) for g, v in dirn.items()})
# waterfall: grade-2 tumour, G3-like, P(G3) closest to 0.5 among those with P(G3)>P(G1)
y = load("GSE7390", genes=CG)[1]
best = None
for f in F:
    for i, s in enumerate(f["te"]):
        if y[s] != 1: continue
        p = f["proba_te"][i]
        if p[2] > p[0] and p[2] < 0.75:
            score = abs(p[2] - 0.5)
            if best is None or score < best[0]: best = (score, f, i, s)
_, f, i, s = best
wf = {"sample": A["GSE7390"]["samples"][s], "base": float(f["shap_base"][2]), "final": float(f["proba_te"][i][2]),
      "pG1": float(f["proba_te"][i][0]), "contrib": [(str(g), int(f["Dte"][i, j]), float(f["shap"][i, j, 2])) for j, g in enumerate(f["genes"])]}
print("waterfall", wf["sample"], round(wf["base"],3), "->", round(wf["final"],3), "pG1", round(wf["pG1"],3))
for c in sorted(wf["contrib"], key=lambda t: -abs(t[2])): print("   ", c[0], "bin", c[1], round(c[2],3))
# permutation: conditional on selection, with 95% CI across folds, for genes selected in >=10% folds
FA = [pickle.load(open(f,"rb")) for f in sorted(glob.glob("folds_GSE7390/r*_k*.pkl"))]
perm = {}
for f in FA:
    for j, g in enumerate(f["genes"]): perm.setdefault(g, []).append((f["perm_grouped"][j], f["perm_single"][j], f["groups"][j], (f["groups"]==f["groups"][j]).sum()))
fr = A["GSE7390"]["freq"]; U = list(fr[fr >= 0.10].index)
pt = []
for g in U:
    v = np.array([(a, b, d) for a, b, _, d in perm[g]])
    m, sd, n = v[:,0].mean(), v[:,0].std(ddof=1), len(v)
    pt.append((g, round(fr[g],2), n, round(100*m,2), round(100*1.96*sd/np.sqrt(n),2), round(100*v[:,1].mean(),2), round(v[:,2].mean(),2)))
pt.sort(key=lambda t: -t[3])
print("perm (gene, freq, nsel, grouped drop %, ±95%, single drop %, mean group size):"); [print("  ", t) for t in pt]
fr_, ranks, CD = S["fried"]
pickle.dump({"prolif": out, "shap_top": top, "sm": sm.loc[top], "g2share": g2share.loc[top], "dirn": dirn, "bees": bees, "wf": wf, "perm": pt, "ranks": ranks, "CD": CD}, open("xai_fig.pkl","wb"))